# SharePoint to Unity Catalog: Connection Setup and Document Ingestion

This notebook walks through the end-to-end setup to ingest documents from Microsoft SharePoint into Unity Catalog, where they can be used downstream by Databricks Knowledge Assistant, AI/BI Dashboards, or any other Databricks workload.

The flow is broken into five parts:

1. Register an application in Microsoft Entra ID and capture its credentials
2. Grant the application read access to specific SharePoint sites using `Sites.Selected`
3. Create a Unity Catalog connection that references the application's credentials
4. Grant Databricks users `USE CONNECTION` on the new connection
5. Create a Delta landing table and ingest documents from a SharePoint location

The result is a Delta table in Unity Catalog containing the raw bytes and metadata of every document, ready for downstream consumption.

## Prerequisites

Before starting, confirm you have access to:

- **Microsoft Entra ID admin access** with permission to register applications and grant API admin consent in the tenant.
- **SharePoint admin access** with permission to grant per-site application permissions (required for `Sites.Selected`).
- **Databricks workspace** with Unity Catalog enabled and serverless compute available.
- **Destination catalog and schema** in Unity Catalog where ingested documents will land. You will need `USE CATALOG`, `USE SCHEMA`, and `CREATE TABLE` privileges on these.
- **SharePoint site URLs** for the corpora you intend to ingest. Each will look like `https://<tenant>.sharepoint.com/sites/<site-name>`.

Have a secure scratch space ready (password manager or scratch text file you will discard) to paste the application's client secret value between steps.

## Part 1: Register an application in Microsoft Entra ID

This creates the service principal that Databricks will authenticate as when reading from SharePoint.

### 1.1 Register the application

1. Navigate to the [Azure portal](https://portal.azure.com).
2. Open **Microsoft Entra ID** > **App registrations**.
3. Click **+ New registration**.
4. Name the application clearly, for example: `Databricks SharePoint Connector`.
5. Under **Supported account types**, select `Accounts in this organizational directory only (single tenant)`.
6. Leave the **Redirect URI** field blank. No redirect URI is required for the OAuth M2M flow used by this connector.
7. Click **Register**.

### 1.2 Capture the application identifiers

From the application's **Overview** page, copy the following values:

- **Application (client) ID** — maps to the `Client ID` field in the Unity Catalog connection.
- **Directory (tenant) ID** — maps to the `Tenant ID` field in the Unity Catalog connection.

### 1.3 Create a client secret

1. Open **Certificates & secrets**.
2. Click **+ New client secret**.
3. Provide a description (e.g., `Databricks UC connector`) and choose an expiration. A 24-month lifetime is typical; calendar the expiration as a rotation reminder.
4. Click **Add**.

The new secret will be displayed with two columns:

- **Value** — the credential. Copy this immediately; Azure will only display this value once.
- **Secret ID** — an internal identifier for the secret record. This is NOT the credential. Do not use it in the Unity Catalog connection.

### 1.4 Grant the application API permissions

1. Open **API permissions**.
2. Click **+ Add a permission** > **Microsoft Graph** > **Application permissions**.
3. Search for and add **`Sites.Selected`**. This permission grants no site access by default; site access must be explicitly granted per-site in Part 2.
4. Click **Grant admin consent for [tenant]** at the top of the permissions table. Confirm in the dialog. The permission status must show as `Granted` before proceeding.

## Part 2: Grant the application read access to specific SharePoint sites

With `Sites.Selected`, the application has no SharePoint access by default. A SharePoint administrator must explicitly grant the application read access on each site you intend to ingest from.

The simplest UI path uses [Microsoft Graph Explorer](https://developer.microsoft.com/en-us/graph/graph-explorer), Microsoft's hosted web tool for executing Graph API calls.

### 2.1 Sign in to Graph Explorer

1. Open https://developer.microsoft.com/en-us/graph/graph-explorer.
2. Click **Sign in to Graph Explorer** in the top-right.
3. Sign in as an administrator with SharePoint administrative rights.
4. When prompted, consent to the permissions Graph Explorer requests on your behalf.

### 2.2 Retrieve the site ID for each SharePoint site

For each site you intend to ingest from:

1. Set the HTTP method dropdown to **GET**.
2. In the URL field, enter (substituting your tenant and site name):

   ```
   https://graph.microsoft.com/v1.0/sites/<tenant>.sharepoint.com:/sites/<site-name>
   ```

3. Click **Run query**.
4. In the response panel, copy the value of the `id` field. It will be a comma-separated string in the form:

   ```
   <tenant>.sharepoint.com,<guid>,<guid>
   ```

   Save this string. You will use it in the next step.

### 2.3 Grant the application read access to the site

For each site ID you collected:

1. Set the HTTP method dropdown to **POST**.
2. In the URL field, enter (substituting the site ID):

   ```
   https://graph.microsoft.com/v1.0/sites/<site-id>/permissions
   ```

3. Open the **Request body** tab and paste the following (substituting your application's client ID):

   ```json
   {
     "roles": ["read"],
     "grantedToIdentitiesV2": [
       {
         "application": {
           "id": "<application-client-id>",
           "displayName": "Databricks SharePoint Connector"
         }
       }
     ]
   }
   ```

4. Click **Run query**. A `201 Created` response indicates the grant succeeded.

Repeat for every SharePoint site Databricks should be able to read.

### A note on SharePoint scoping

SharePoint stores everything inside a "site," even when the structure is not obvious to end users. Microsoft Teams channels are backed by SharePoint sites. OneDrive folders are backed by per-user sites. Standalone document libraries live inside a parent site. If your source content is in a Teams channel, identify the SharePoint site that backs it and grant access at that level.

`Sites.Selected` is granted only at the site level. Folder-level scoping is handled later in the SQL ingestion, where you can point `read_files()` at a specific subfolder of an authorized site.

## Part 3: Create the Unity Catalog connection

This step creates a Unity Catalog connection that holds the application's credentials. Once created, Databricks pipelines and SQL queries reference the connection by name without handling the underlying secret directly.

1. In the Databricks left sidebar, click **Catalog**.
2. Click **+ Add** > **Add connection**.
3. Complete the form with the following values:

   | Field | Value |
   |---|---|
   | Connection name | `parkview_sharepoint` (or a name of your choice) |
   | Connection type | `Microsoft SharePoint` |
   | Authentication type | `OAuth Machine to Machine` |
   | Client ID | The Application (client) ID from Part 1.2 |
   | Client secret | The Value (not the Secret ID) from Part 1.3 |
   | Domain | `https://<tenant>.sharepoint.com` (tenant root only, no path) |
   | Tenant ID | The Directory (tenant) ID from Part 1.2 |

4. Click **Create connection**. A green status indicator confirms a successful authentication test.

If authentication fails, common causes are:

- Admin consent was not granted on the Entra ID application (Part 1.4).
- The Client secret field contains the Secret ID instead of the Value.
- The application has not been granted access to any sites in Part 2.
- The Domain field contains a path segment beyond the tenant root.

## Part 4: Grant USE CONNECTION

The connection now exists, but no Databricks user can reference it until you grant permission.

1. Open the connection page in Catalog Explorer.
2. Click the **Permissions** tab.
3. Click **Grant**.
4. Specify the grantee (a user, group, or service principal). For development, grant to the engineer or team that will build the ingestion pipeline. For production, grant to a dedicated service principal.
5. Check **USE CONNECTION**.
6. Click **Grant**.

Avoid granting USE CONNECTION to broad groups such as `account users` or `all workspace users`. SharePoint corpora frequently contain sensitive content; the connection's grantee list should match the audience allowed to ingest those documents.

You will also need standard Unity Catalog privileges (`USE CATALOG`, `USE SCHEMA`, `CREATE TABLE`) on the destination catalog and schema before running the SQL cells that follow. Those grants are scoped to the destination objects, not to the connection.

## Part 5: Ingest documents into a Delta landing table

The remaining cells create a Delta landing table in Unity Catalog and ingest documents from a SharePoint location into it.

Before running, update the placeholders in each SQL cell:

| Placeholder | Description |
|---|---|
| `<CATALOG>` | Destination Unity Catalog (e.g., `parkview_catalog`) |
| `<SCHEMA>` | Destination schema (e.g., `bronze`) |
| `<TABLE>` | Landing table name (e.g., `data_governance_docs`) |
| `<USE_CASE>` | Friendly label for the table comment (e.g., `Data Governance KA`) |
| `<SHAREPOINT_URL>` | Source SharePoint site, library, or folder URL |
| `<CONNECTION_NAME>` | The connection name from Part 3 (e.g., `parkview_sharepoint`) |
| `<FILE_PATTERN>` | Glob filter for file types (e.g., `*.pdf` or `*.{pdf,docx,pptx}`) |

For multiple corpora (e.g., one table for the Data Governance use case, a separate table for the Parkview GenAI use case), run the two SQL cells once per corpus, varying the table name and SharePoint URL each time.

In [0]:
-- Part 5a: Create the landing Delta table for SharePoint documents
-- Update placeholders before running.

CREATE TABLE IF NOT EXISTS <CATALOG>.<SCHEMA>.<TABLE> (
  content BINARY COMMENT 'Raw file bytes ingested from SharePoint',
  metadata STRUCT<
    file_path: STRING,
    file_name: STRING,
    file_size: BIGINT,
    file_modification_time: TIMESTAMP
  > COMMENT 'File metadata captured at ingestion time'
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'Documents ingested from SharePoint via Lakeflow Connect for <USE_CASE>';

In [0]:
-- Part 5b: Ingest documents from SharePoint into the landing table
-- Re-running this cell is safe; the WHERE clause skips files already present.

INSERT INTO <CATALOG>.<SCHEMA>.<TABLE>
SELECT
  content,
  named_struct(
    'file_path',              path,
    'file_name',              regexp_extract(path, '([^/]+)$', 1),
    'file_size',              CAST(length AS BIGINT),
    'file_modification_time', modificationTime
  ) AS metadata
FROM read_files(
  "<SHAREPOINT_URL>",
  `databricks.connection` => "<CONNECTION_NAME>",
  format                  => "binaryFile",
  pathGlobFilter          => "<FILE_PATTERN>"
)
WHERE path NOT IN (
  SELECT metadata.file_path
  FROM <CATALOG>.<SCHEMA>.<TABLE>
);

In [0]:
-- Part 5c: Verify what landed
-- Row count, total size, file age range, and a sample of ingested files.

SELECT COUNT(*) AS file_count,
       ROUND(SUM(metadata.file_size) / 1024.0 / 1024.0, 2) AS total_size_mb,
       MIN(metadata.file_modification_time) AS oldest_file_mtime,
       MAX(metadata.file_modification_time) AS newest_file_mtime
FROM <CATALOG>.<SCHEMA>.<TABLE>;

SELECT metadata.file_name,
       metadata.file_path,
       metadata.file_size,
       metadata.file_modification_time
FROM <CATALOG>.<SCHEMA>.<TABLE>
ORDER BY metadata.file_modification_time DESC
LIMIT 10;

## Next steps

With documents landed in Unity Catalog, the table is ready to serve as input for:

- **Knowledge Assistant** in Agent Bricks: point a Knowledge Assistant agent at the Delta table to deliver a hosted RAG chatbot with citation handling, no additional code required.
- **AI/BI Dashboards**: analyze ingestion volume, file type distribution, and freshness over time.
- **Lakeflow Pipelines**: schedule incremental refreshes on a cadence appropriate to your source SharePoint update frequency.
- **Custom Databricks Apps**: build purpose-built UIs that combine RAG retrieval with workflow logic.

References:

- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)
- [SharePoint ingestion overview](https://docs.databricks.com/aws/en/ingestion/sharepoint)
- [OAuth M2M for SharePoint](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/sharepoint-source-setup-m2m)